In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

print(path)
print(os.listdir(path)) #Looking for train and test paths
path = os.path.join(path,"PlantVillage")
print(path)
print(os.listdir(path))

In [ ]:
#imports
from torch.utils.data import Dataset
from PIL import Image
import glob
import os
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm



In [ ]:
# Custome Dataset
class Potato_Disease_Dataset(Dataset):
  def __init__(self,root,transform):
    self.root = root
    self.transform = transform

    self.classes = sorted(os.listdir(root))

    self.class_label = {classname : i for i,classname in enumerate(self.classes)}

    self.image_paths = []
    self.labels = []

    for classname in self.classes:
      class_dir = os.path.join(root,classname)
      class_idx = self.class_label[classname]

      for img_name in os.listdir(class_dir):
        img_path = os.path.join(class_dir,img_name)
        self.image_paths.append(img_path)
        self.labels.append(class_idx)


  def __len__(self):
    return len(self.image_paths)

  def __getitem__(self,idx):
    img_path = self.image_paths[idx]
    img = Image.open(img_path).convert('RGB')

    label = self.labels[idx]

    img = self.transform(img)

    return img,label


In [ ]:
# Use RandomRotation(15) Augmentation on the training dataset + Reize the images to 32x32
train_transform = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize images
    transforms.RandomRotation(15),  # Rotate images randomly within ±15 degrees
    transforms.ToTensor(),  # Convert to tensor
])

test_transform = transforms.Compose([  #No augmentation in testing
    transforms.Resize((32, 32)),  # Resize images
    transforms.ToTensor(),  # Convert to tensor
])

In [ ]:
# Create the training and testing datasets, and their DataLoaders

train_path = os.path.join(path,"train")
test_path = os.path.join(path,"test")
print(os.listdir(train_path))
#Creating datasets
train_dataset = Potato_Disease_Dataset(train_path,transform = train_transform)
test_dataset = Potato_Disease_Dataset(test_path,transform = test_transform)

# #Creating dataloaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

In [ ]:
# Display some sample images with their labels
import random
import numpy as np
import matplotlib.pyplot as plt


fig, axes = plt.subplots(1, 5, figsize=(15, 5))

classes = ['Potato___healthy', 'Potato___Late_blight', 'Potato___Early_blight']
for i in range(5):
    idx = np.random.randint(0, len(train_dataset))
    img, label = train_dataset.__getitem__(idx)

    img_np = img.numpy().transpose(1, 2, 0)

    img_np = np.clip(img_np, 0, 1)

    axes[i].imshow(img_np)
    axes[i].set_title(f'Class: {classes[label]}')
    axes[i].axis('off')

plt.show()

In [ ]:
# Write your code here
class MyCNN(nn.Module):
  def __init__(self,num_classes = 3):
    super().__init__()
    self.features = nn.Sequential(

            nn.Conv2d(3, 8, kernel_size=3, padding=1),  # [B,8,32,32]
            nn.LeakyReLU(),
            nn.MaxPool2d(2),                             # [B,8,16,16]

            nn.Conv2d(8, 16, kernel_size=3, padding="same"),  # [B,16,16,16]
            nn.LeakyReLU(),
            #nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding="same"),  # [B,32,16,16]
            nn.LeakyReLU(),
            nn.MaxPool2d(2),                                # [B,32,8,8]

            nn.Conv2d(32, 64, kernel_size=3, padding="same"),  # [B,64,8,8]
            nn.LeakyReLU(),
            #nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),  # [B,128,8,8]
            nn.LeakyReLU(),
            #nn.MaxPool2d(2),                            #[B,128,4,4]
        )

    self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*8*8, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

  def forward(self, x):
      x = self.features(x)
      x = self.classifier(x)
      return x


In [ ]:
#Setting up the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MyCNN().to(device)

print(model)

In [ ]:
# Write your code here
def accuracy_from_logits(logits, labels):
    preds = torch.argmax(logits, dim=1)
    return (preds == labels).float().mean().item()

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_acc = 0.0, 0.0

    for images,labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        logits = model(images) #Get logits
        loss = criterion(logits, labels) #CEL already have softmax

        loss.backward() #Backward pass

        optimizer.step() #Update weights

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits.detach(), labels)

    return total_loss / len(loader), total_acc / len(loader)

def evaluate(model, loader, criterion):
    model.eval() #Not updating weights
    total_loss, total_acc = 0.0, 0.0

    with torch.no_grad():
        for images,labels in loader:
            images, labels = images.to(device), labels.to(device)

            logits = model(images) #Predict

            loss = criterion(logits, labels) #Calculate loss

            total_loss += loss.item()
            total_acc += accuracy_from_logits(logits, labels)

    return total_loss / len(loader), total_acc / len(loader)

In [ ]:
# Write your code here
#device already set

criterion = nn.CrossEntropyLoss() #Defining loss function, multiclass classification
learning_rate = 0.001
optimizer = optim.AdamW(model.parameters(), lr= learning_rate)
epochs = 10 #Number of epochs


#For visualization
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []

# Training
for epoch in range(epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, optimizer,criterion)
    test_loss, test_accuracy = evaluate(model, test_loader, criterion)

    # Store metrics
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accuracies.append(train_accuracy)
    test_accuracies.append(test_accuracy)

    print(f"Epoch {epoch+1}/{epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={test_loss:.4f}, Val Accuracy={test_accuracy:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, epochs+1), test_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, epochs+1), test_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()

In [ ]:
# Write your code here
